# Dateiimport: ein Beispiel

Wir lesen die drei Dateitypen eines Experiments und sehen uns das Ergebnis an.
Die Rohdaten werden nicht verändert. Zeitstempel sind in Millisekunden.

Das Notebook im Ordner `notebooks` oder im Projektordner starten.
Benötigt werden die Pakete aus `requirements.txt`.


In [ ]:
from pathlib import Path
import sys

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent

sys.path.insert(0, str(project))
from src.spectral_sensor.io import read_sensor_csv, read_marker_rows, read_protocol

data = project / "experiments/E02_20x15_mag_steering/data"
sensor_file = data / "SpectralSensor 1_20260708_113243.csv"
marker_file = data / "Markers_20260708_113243.csv"
protocol_file = data / "Protocol.md"


## Sensorwerte

Jede Zeile ist ein Messpunkt. Die Spaltennamen und Werte bleiben wie in der CSV.


In [ ]:
sensor = read_sensor_csv(sensor_file)
print(f"{len(sensor)} Messpunkte")
sensor.head()


## Marker

Ein Marker enthält einen Zeitstempel und einen Beschreibungstext.
`parse_error` zeigt fehlerhafte Zeilen. Die Bedeutung der Marker wird erst im Analyzer ausgewertet.


In [ ]:
markers = read_marker_rows(marker_file)
markers.head()


## Protokoll

Diese Angaben beschreiben das Experiment und die Herkunft der Messung.


In [ ]:
metadata = read_protocol(protocol_file)
metadata


## Vergleich mit dem bisherigen Import

Die Originalversion ist als feste Vergleichsdatei in `validation/fixtures/io_before_refactor.py` gespeichert.
Wir prüfen für diese Dateien, ob der Analyzer nach der Auslagerung dieselben Tabellen liefert – einschließlich Datentypen und Reihenfolge.


In [ ]:
from pandas.testing import assert_frame_equal
from validation.fixtures.io_before_refactor import SpectralExperimentAnalyzer as Before
from spectral_experiment_analyzer import SpectralExperimentAnalyzer as After

assert_frame_equal(
    Before._read_sensor(sensor_file),
    After._read_sensor(sensor_file),
    check_exact=True,
)
assert_frame_equal(
    Before._read_markers(marker_file),
    After._read_markers(marker_file),
    check_exact=True,
)
assert Before._read_protocol(protocol_file) == After._read_protocol(protocol_file)

print("OK: Der Import liefert unveränderte Ergebnisse.")


Das ist ein Importvergleich für ein Experiment, kein Test der gesamten Auswertung.

**Noch offen:** Die automatische Dateisuche verwendet bisher `Spectralsensor`, die vorhandenen Dateien heißen `SpectralSensor`. Deshalb wählen wir die Dateien hier ausdrücklich aus. Diese bestehende Abweichung wurde nicht nebenbei geändert.
